# Cats vs Dogs Image Classification 🐱🐶

A hands-on **2D CNN** notebook that classifies photos as *cat* or *dog*. This is the classic first computer-vision project.

We'll cover the full pipeline:
1. Download the Cats-vs-Dogs dataset
2. Load images efficiently with `image_dataset_from_directory`
3. Add data augmentation to reduce overfitting
4. Build and train a CNN from scratch
5. Evaluate and view predictions
6. **Bonus:** transfer learning with MobileNetV2 for a big accuracy jump

> Tip: Training is *much* faster on a GPU. In Google Colab go to **Runtime → Change runtime type → GPU**.


## 1. Setup

In [ ]:
# !pip install tensorflow matplotlib

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

tf.random.set_seed(42)
print("TensorFlow version:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))

## 2. Download the dataset

We use the Microsoft Cats-vs-Dogs dataset (~786 MB, ~25,000 images). Keras can download and unzip it for us.

The archive contains a few corrupt images, so we filter those out before training (a common real-world data-cleaning step).

In [ ]:
import pathlib

url = "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"
zip_path = tf.keras.utils.get_file("cats_and_dogs.zip", origin=url, extract=True)

base_dir = pathlib.Path(zip_path).parent / "PetImages"
print("Data directory:", base_dir)
print("Classes:", [p.name for p in base_dir.iterdir() if p.is_dir()])

In [ ]:
# Remove corrupt images that aren't valid JPEGs
import imghdr

num_removed = 0
for folder in ("Cat", "Dog"):
    folder_path = base_dir / folder
    for fname in os.listdir(folder_path):
        fpath = folder_path / fname
        try:
            with open(fpath, "rb") as f:
                is_jfif = b"JFIF" in f.peek(10)
            if not is_jfif:
                num_removed += 1
                os.remove(fpath)
        except Exception:
            num_removed += 1
            try: os.remove(fpath)
            except OSError: pass

print(f"Removed {num_removed} corrupt/invalid images")

## 3. Load images as tf.data datasets

`image_dataset_from_directory` reads images straight from the folders, resizes them, batches them, and infers labels from the folder names (`Cat` → 0, `Dog` → 1).

We split 80% training / 20% validation.

In [ ]:
IMG_SIZE = (180, 180)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    base_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    base_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names
print("Class names:", class_names)

### Peek at the data
Always look at your images before training — it catches labeling and loading bugs early.

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.tight_layout(); plt.show()

### Performance: prefetching

`prefetch` lets the CPU prepare the next batch while the GPU trains on the current one, removing an I/O bottleneck.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

## 4. Data augmentation

Randomly flipping, rotating, and zooming images creates new variations on the fly. The model sees a slightly different image each epoch, which **reduces overfitting** and helps it generalize.

In [ ]:
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="data_augmentation")

# Visualize augmentation on one image
plt.figure(figsize=(10, 10))
for images, _ in train_ds.take(1):
    first = images[0]
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        aug = data_augmentation(tf.expand_dims(first, 0))
        plt.imshow(aug[0].numpy().astype("uint8"))
        plt.axis("off")
plt.tight_layout(); plt.show()

## 5. Build a CNN from scratch

Architecture intuition:
- **Rescaling** normalizes pixels from 0–255 to 0–1 so training is stable.
- Each **Conv2D + MaxPooling** block detects patterns (edges → textures → shapes) at increasing abstraction while shrinking the spatial size.
- **GlobalAveragePooling** collapses the feature maps into one vector.
- **Dropout** randomly zeroes activations during training to fight overfitting.
- A single **sigmoid** unit outputs P(dog).

In [ ]:
model = models.Sequential([
    layers.Input(shape=IMG_SIZE + (3,)),
    data_augmentation,
    layers.Rescaling(1.0 / 255),

    layers.Conv2D(32, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),

    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## 6. Train

We train for 10 epochs. Expect roughly **80–85%** validation accuracy from a scratch-built CNN — good, and a strong baseline before the transfer-learning section.

*(Reduce `EPOCHS` if you're on CPU and just want to see it run.)*

In [ ]:
EPOCHS = 10

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
)

In [ ]:
def plot_history(history):
    acc = history.history["accuracy"]
    val_acc = history.history["val_accuracy"]
    loss = history.history["loss"]
    val_loss = history.history["val_loss"]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(acc, label="train"); axes[0].plot(val_acc, label="val")
    axes[0].set_title("Accuracy"); axes[0].set_xlabel("epoch"); axes[0].legend()
    axes[1].plot(loss, label="train"); axes[1].plot(val_loss, label="val")
    axes[1].set_title("Loss"); axes[1].set_xlabel("epoch"); axes[1].legend()
    plt.tight_layout(); plt.show()

plot_history(history)

## 7. Look at predictions

In [ ]:
plt.figure(figsize=(12, 12))
for images, labels in val_ds.take(1):
    preds = model.predict(images, verbose=0).ravel()
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        pred_label = class_names[int(preds[i] > 0.5)]
        true_label = class_names[labels[i]]
        conf = preds[i] if preds[i] > 0.5 else 1 - preds[i]
        color = "green" if pred_label == true_label else "red"
        plt.title(f"{pred_label} ({conf:.0%})\ntrue: {true_label}", color=color)
        plt.axis("off")
plt.tight_layout(); plt.show()

## 8. Bonus: Transfer learning with MobileNetV2 🚀

Instead of learning from scratch, we reuse **MobileNetV2** — a network already trained on ImageNet (1.4M images). Its early layers already know edges, textures, and shapes, so we just train a small classifier head on top.

This usually pushes accuracy to **97%+** with far less training. This is how most real-world image classifiers are built.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,      # drop ImageNet's 1000-class head
    weights="imagenet",
)
base_model.trainable = False   # freeze the pretrained weights

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)  # scales to [-1, 1]
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

transfer_model = models.Model(inputs, outputs)
transfer_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
transfer_model.summary()

In [ ]:
transfer_history = transfer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
)
plot_history(transfer_history)

## 9. Save the model & next steps

```python
transfer_model.save("cats_vs_dogs_model.keras")
# Reload later with:
# loaded = tf.keras.models.load_model("cats_vs_dogs_model.keras")
```

**Recap**
- Loaded and cleaned a real image dataset.
- Used augmentation + dropout to fight overfitting.
- Built a CNN from scratch, then beat it easily with transfer learning.

**Try next**
1. **Fine-tune**: unfreeze the top layers of `base_model` (`base_model.trainable = True`) and retrain with a *very low* learning rate (e.g. `1e-5`) for a further accuracy bump.
2. Predict on **your own** cat/dog photo:
   ```python
   img = tf.keras.utils.load_img("my_pet.jpg", target_size=IMG_SIZE)
   arr = tf.expand_dims(tf.keras.utils.img_to_array(img), 0)
   p = transfer_model.predict(arr)[0][0]
   print("Dog" if p > 0.5 else "Cat", f"({p:.2f})")
   ```
3. Add a **confusion matrix** with `sklearn.metrics` to see which class is harder.
4. Try other pretrained backbones: `EfficientNetB0`, `ResNet50`.

Great work — you've built a real image classifier! 🎉